# Fiche 7 — Valider un modèle : entraînement, validation, test… et fuite de données

| | |
|---|---|
| 🎯 **Objectif** | Expliquer le rôle des trois jeux de données, découper sans fuite, et reconnaître une évaluation faussée par des images déjà vues |
| ⏱️ **Durée** | 50 min |
| 🧰 **Prérequis** | Fiche 6 |
| 🔗 **Dans le projet** | Étape 5 (méthode de validation : jeux d'entraînement, de validation et de test) |

> **L'idée clé.** Un test ne mesure quelque chose que si le modèle **n'a jamais vu** les images du test. Sinon, c'est un examen dont l'élève connaissait les questions : la note est bonne et ne prouve rien.

## 1. Trois jeux, trois rôles

| Jeu | Analogie | Rôle |
|---|---|---|
| **Entraînement** | les exercices du cours | le modèle apprend dessus |
| **Validation** | les examens blancs | on choisit les réglages, on décide quand arrêter d'apprendre |
| **Test** | l'examen final, sous enveloppe | on mesure **une seule fois**, à la fin, sur du jamais vu |

Dans ce projet, on n'entraîne rien : le modèle est déjà entraîné, donc nos 50 images ne servent **que de test**. Les
trois jeux deviendront nécessaires si ModeTrends veut un jour affiner le modèle sur ses propres photos.

## 2. Démonstration : un « modèle » qui apprend par cœur

On fabrique le pire élève possible : il mémorise le masque de chaque image d'entraînement (il reconnaît l'image à
son empreinte) et, devant une image inconnue, répond le masque « moyen » qu'il a vu. On l'entraîne sur 40 images et
on l'évalue sur ces 40 images, puis sur les 10 autres.

In [1]:
import hashlib

import numpy as np
from PIL import Image

from segvet import evaluation as ev
from segvet import io

pairs = io.load_pairs()
rng = np.random.default_rng(0)
ordre = rng.permutation(len(pairs))
entrainement, test = ordre[:40], ordre[40:]


def empreinte(chemin):
    return hashlib.sha256(chemin.read_bytes()).hexdigest()


print("Empreinte de image_0 :", empreinte(pairs[0][0])[:16], "…")
print("Même fichier relu     :", empreinte(pairs[0][0])[:16], "… → identique")
print("Autre image           :", empreinte(pairs[1][0])[:16], "… → différente")

Empreinte de image_0 : 7f45efa597869f7d …
Même fichier relu     : 7f45efa597869f7d … → identique
Autre image           : 3005d08991843dc8 … → différente


Une **empreinte** (*hash*) est un code calculé à partir des octets d'un fichier : même fichier, même code ; le moindre pixel changé donne un tout autre code. C'est ce qui permet au « modèle » ci-dessous de reconnaître une image déjà vue.

In [2]:
class ParCoeur:
    def fit(self, indices):
        self.memoire = {empreinte(pairs[i][0]): io.read_mask(pairs[i][1]) for i in indices}
        return self

    def predict(self, chemin):
        h, w = np.array(Image.open(chemin)).shape[:2]
        if empreinte(chemin) in self.memoire:
            return self.memoire[empreinte(chemin)]
        # Image inconnue : on devine un haut au centre et un pantalon dessous, comme « en moyenne »
        devine = np.zeros((h, w), np.uint8)
        devine[h // 5 : h // 2, w // 3 : 2 * w // 3] = 4
        devine[h // 2 : 9 * h // 10, w // 3 : 2 * w // 3] = 6
        return devine

In [3]:
modele = ParCoeur().fit(entrainement)
for nom, indices in [("images déjà vues (entraînement)", entrainement), ("images jamais vues (test)", test)]:
    cms = ev.image_confusions([io.read_mask(pairs[i][1]) for i in indices], [modele.predict(pairs[i][0]) for i in indices])
    print(f"{nom:<33}: mIoU vêtements = {ev.clothing_miou(cms.sum(axis=0)):.2f}")

images déjà vues (entraînement)  : mIoU vêtements = 1.00
images jamais vues (test)        : mIoU vêtements = 0.08


**Lecture.** Note parfaite sur ce qu'il a appris, note catastrophique sur le reste. Si on l'avait évalué sur ses
images d'entraînement, on l'aurait cru parfait.

## ⚠️ Piège : c'est exactement ce qui arrive dans ce projet

Le modèle `segformer_b3_clothes` a été affiné sur un jeu public de 17 706 photos annotées (ATR,
`mattmdjaga/human_parsing_dataset`). En comparant les masques pixel à pixel, on y retrouve **les 50 masques de test,
à l'identique**. Le code d'entraînement publié par l'auteur ne met que 0,6 % des images de côté pour le test : on
s'attend donc à ce que 49 ou 50 de nos images aient servi à l'entraînement.

Le script de vérification (il télécharge 800 Mo) compare des **empreintes** de masques, comme le `ParCoeur`
ci-dessus :

```python
# pour chaque ligne du jeu ATR (fichiers parquet du dépôt Hugging Face) :
masque = np.array(Image.open(io.BytesIO(ligne["mask"]["bytes"])))
cle = hashlib.md5(masque.tobytes() + str(masque.shape).encode()).hexdigest()
if cle in empreintes_de_nos_50_masques: trouves += 1      # résultat : 50 sur 50
```

Conséquence : le score que tu obtiendras sur ces 50 images est une **estimation optimiste, qui ne mesure pas la
généralisation**. Il prouve seulement que le modèle ne rate pas grossièrement des images qu'il connaît.

## 3. Découper sans fuite : par personne, pas par photo

Pour un futur jeu ModeTrends, deux photos de la **même influenceuse** dans la même tenue se ressemblent trop : si
l'une est en entraînement et l'autre en test, le test redevient un examen connu. On découpe donc par **groupe**
(l'influenceur), pas par photo. Démonstration sur un petit catalogue imaginaire :

In [4]:
import pandas as pd

catalogue = pd.DataFrame({
    "photo": [f"p{i:02d}" for i in range(12)],
    "influenceur": ["ana", "ana", "ana", "ben", "ben", "chloe", "chloe", "chloe", "chloe", "dina", "eli", "eli"],
})
personnes = catalogue["influenceur"].unique()
melange = np.random.default_rng(1).permutation(personnes)
role = {p: "test" if p in melange[:1] else "validation" if p in melange[1:2] else "entraînement" for p in melange}
catalogue["jeu"] = catalogue["influenceur"].map(role)
print(catalogue.groupby("jeu")["influenceur"].unique())
print("Une personne dans deux jeux ?", catalogue.groupby("influenceur")["jeu"].nunique().max() > 1)

jeu
entraînement    [ben, chloe, dina]
test                         [eli]
validation                   [ana]
Name: influenceur, dtype: object
Une personne dans deux jeux ? False


### ✍️ À toi de jouer

Reprends `catalogue` et fais un découpage **par photo** (12 photos mélangées : 8 entraînement, 2 validation, 2 test). Compte combien d'influenceurs du jeu de test ont aussi des photos en entraînement.

In [5]:
# Ta réponse ici

<details>
<summary><b>💡 Voir une solution</b> (après avoir cherché !)</summary>

```python
melange_photos = np.random.default_rng(1).permutation(len(catalogue))
jeu = np.array(["entraînement"] * 8 + ["validation"] * 2 + ["test"] * 2)
catalogue["jeu_par_photo"] = jeu[np.argsort(melange_photos)]
en_test = set(catalogue.query("jeu_par_photo == 'test'")["influenceur"])
en_entrainement = set(catalogue.query("jeu_par_photo == 'entraînement'")["influenceur"])
print(en_test & en_entrainement)   # le plus souvent, au moins une personne est des deux côtés
```

**Pourquoi ?** avec un découpage par photo, une personne qui a 3 ou 4 photos a de grandes chances d'apparaître des deux côtés : le modèle peut reconnaître la tenue plutôt que comprendre la segmentation. Le découpage par groupe l'interdit par construction.

</details>

## ✅ À retenir

1. Entraînement pour apprendre, validation pour régler, test pour mesurer **une fois** sur du jamais vu.
2. Un modèle évalué sur ses images d'entraînement paraît meilleur qu'il n'est (le `ParCoeur` obtient 1,00).
3. Les 50 images du projet font partie des données d'entraînement du modèle : leur score est **optimiste**.
4. On vérifie l'absence de doublons entre jeux avec des **empreintes** (hash) d'images ou de masques.
5. On découpe **par personne** (groupe), pas par photo, pour que le test reste inconnu.